# 40 — kNN and the curse of dimensionality

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/omarahmedZa/ai-ml-course/blob/main/notebooks/040-knn-curse-of-dimensionality.ipynb)

Nearest neighbours vote. This notebook computes the five distances from the lesson, runs the vote for `k = 1, 3, 5`, draws the neighbourhood, and then measures how adding noise features destroys the meaning of "nearest".

In [ ]:
import numpy as np  # arrays and distances
X = np.array([[1., 1.], [2., 3.], [4., 2.], [3., 4.], [0., 1.]])  # training points
y = np.array(["A", "B", "A", "B", "A"])  # their labels
q = np.array([2., 2.])  # the query
d = np.linalg.norm(X - q, axis=1)  # Euclidean distance to each point
print("distances", np.round(d, 3))  # [1.414 1. 2. 2.236 2.236]
order = np.argsort(d)  # nearest first
print("order", order)  # [1 0 2 3 4]
for k in (1, 3, 5):  # three neighbourhood sizes
    labels, counts = np.unique(y[order[:k]], return_counts=True)  # tally votes
    print(k, labels[counts.argmax()])  # B, then A, then A

## The vote flips once

With `k = 1` the nearest point `(2, 3)` is a B. With `k = 3` the neighbours are B, A, A, so the vote is A. With `k = 5` the count is A: 3, B: 2, still A. The `k = 1` answer is the fragile one: a single nearby point decides everything.

### The neighbourhood

The dashed circle below has radius 2. It contains three points, which is why `k = 3` uses them — and `(4, 2)` sits exactly on the boundary, the unstable case from the lesson.

In [ ]:
import matplotlib.pyplot as plt  # plotting
plt.figure(figsize=(5, 5))  # square figure
for label, colour in (("A", "tab:blue"), ("B", "tab:orange")):  # one colour per class
    mask = y == label  # rows of this class
    plt.scatter(X[mask, 0], X[mask, 1], color=colour, label=label, s=60)  # points
plt.scatter(q[0], q[1], marker="x", color="black", s=100, label="query")  # query
circle = plt.Circle((q[0], q[1]), 2.0, fill=False, linestyle="--", color="grey")  # radius 2
plt.gca().add_patch(circle)  # draw the circle
plt.gca().set_aspect("equal")  # keep distances honest
plt.xlim(-0.5, 4.5)  # fixed x range
plt.ylim(-0.5, 4.5)  # fixed y range
plt.legend()  # name the marks
plt.title("k = 3 neighbours inside the dashed circle")  # takeaway
plt.show()  # draw it

## When dimensions grow

Add noise features and every distance gains the same kind of contribution, so the gap between the nearest and the farthest point shrinks. At 100 dimensions the axis fraction needed to capture 10% of a cube is `0.1 ** (1 / 100) = 0.977`: a local neighbourhood has become almost the whole space.

### What to do

Scale the features, drop weak ones, reduce with PCA, or use a model that shares strength across dimensions. More rows do not fix distance concentration.

In [ ]:
rng = np.random.default_rng(2)  # reproducible noise
base = np.vstack([X, q])  # stack points and query
noise = rng.normal(0, 1, size=(base.shape[0], 95))  # 95 noise columns
wide = np.hstack([base, noise])  # 97-dimensional version
dist = np.linalg.norm(wide[:-1] - wide[-1], axis=1)  # distances to the query
print("nearest", dist.min())  # some number
print("farthest", dist.max())  # almost the same
print("ratio", dist.min() / dist.max())  # close to 1
print("axis fraction", 0.1 ** (1 / 100))  # 0.977